# Unit 5: Read data and build batches

Read episode rows in order. Build action windows without crossing an episode end. Split whole episodes and fit statistics on training data only.

**Notebook:** learner copy. Work through the cells in order.

In [ ]:
from __future__ import annotations


from typing import TYPE_CHECKING

if TYPE_CHECKING:
    from collections.abc import Mapping
    from collections.abc import Sequence
    from numpy.typing import ArrayLike
    from vla_lab.teaching.records import ACTBatchData
    from vla_lab.teaching.records import EpisodeArrays
    from vla_lab.teaching.records import EpisodeRow
    from vla_lab.teaching.records import TrainingEpisode


from pathlib import Path
import os, sys
WORKSPACE = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'manifests/act_references.yaml').is_file()), None)
if WORKSPACE is None:
    raise FileNotFoundError('Run this notebook inside the VLA course workspace')
sys.path.insert(0, str(WORKSPACE / 'src'))
HF_ROOT = WORKSPACE / '.local-data' / 'hf'
os.environ.setdefault('HF_HUB_CACHE', str(HF_ROOT / 'hub'))
os.environ.setdefault('HF_DATASETS_CACHE', str(HF_ROOT / 'datasets'))
os.environ.setdefault('HF_HUB_OFFLINE', '1')
os.environ.setdefault('HF_DATASETS_OFFLINE', '1')
print('Workspace:', WORKSPACE)


In [ ]:
from __future__ import annotations
import hashlib, json
import numpy as np
from vla_lab.data.stats import TrainingStats
from vla_lab.teaching.data import order_episode_rows, extract_episode_arrays, action_window, split_episode_ids, fit_training_statistics, build_act_batches


## Exercise 1: Order recorded rows

**Purpose:** Read rows in stable episode and frame order.

**Contract:** `order_episode_rows(rows) -> list[rows]`

Rows contain nonnegative integer `episode_index` and `frame_index`; duplicate pairs are invalid.

**Method:** Sort by the two-part key `(episode_index, frame_index)`.

**Pointers:** A frame number alone is not unique across episodes.

<details><summary>Detailed hint</summary>

Keep the original row dictionaries so other dataset columns remain available.

</details>

Estimated time: about 15 minutes.

In [ ]:
def order_episode_rows[RowT: EpisodeRow](rows: Sequence[RowT]) -> list[RowT]:
    raise NotImplementedError('Exercise 1: order rows by episode and frame')

### Check

In [ ]:
rows = [{'episode_index': 1, 'frame_index': 1}, {'episode_index': 0, 'frame_index': 2}]
ordered = order_episode_rows(rows)
assert [(r['episode_index'], r['frame_index']) for r in ordered] == [(0, 2), (1, 1)], 'Rows are not ordered by episode then frame'

### Use your function

In [ ]:
print('First ordered key:', ordered[0]['episode_index'], ordered[0]['frame_index'])

## Exercise 2: Read model columns from raw rows

**Purpose:** Turn real recorded rows into action and state arrays for one full episode.

**Contract:** `extract_episode_arrays(rows) -> {episode_index, frame_indices, actions, states}`

Each row has `episode_index`, `frame_index`, `action` (7), and `observation.state` (8). Sort with Exercise 1 and reject mixed episode IDs.

**Method:** Read the stored action column directly. Do not create a delta action here; make chunks in Exercise 3.

**Pointers:** A frame row contains one saved action. Preserve that action and its pre-action state.

<details><summary>Detailed hint</summary>

Stack each ordered row's `action` and `observation.state` values into a matrix.

</details>

Estimated time: about 15 minutes.

In [ ]:
def extract_episode_arrays(rows: Sequence[EpisodeRow]) -> EpisodeArrays:
    raise NotImplementedError('Exercise 2: extract raw action and state columns')

### Check

In [ ]:
rows = [{'episode_index': 3, 'frame_index': 1, 'action': np.ones(7), 'observation.state': np.ones(8)},
        {'episode_index': 3, 'frame_index': 0, 'action': np.zeros(7), 'observation.state': np.zeros(8)}]
part = extract_episode_arrays(rows)
assert part['frame_indices'].tolist() == [0, 1], 'Raw rows are not ordered'
assert np.allclose(part['actions'][0], 0) and np.allclose(part['states'][1], 1), 'Wrong raw columns were read'

### Use your function

In [ ]:
print('Episode rows:', len(part['actions']), 'action/state:', part['actions'].shape, part['states'].shape)

## Exercise 3: Build a padded action window

**Purpose:** Create a fixed-size chunk and mark its padded tail.

**Contract:** `action_window(actions, *, start, horizon, action_dim=None) -> (window, mask)`

Input has shape `(episode_steps, action_dim)`. Output has `(horizon, action_dim)` and bool mask `(horizon,)`; `True` means padding.

**Method:** Copy only rows from this episode. Fill beyond its end with zeros and mark them padded.

**Pointers:** Do not concatenate episodes before window creation.

<details><summary>Detailed hint</summary>

Initialize zeros and an all-True mask, then overwrite the valid prefix.

</details>

Estimated time: about 15 minutes.

In [ ]:
def action_window(
    episode_actions: ArrayLike, *, start: int, horizon: int, action_dim: int | None = None
) -> tuple[np.ndarray, np.ndarray]:
    raise NotImplementedError('Exercise 2: create an episode-safe action window')

### Check

In [ ]:
window, mask = action_window([[1, 2], [3, 4]], start=1, horizon=3)
assert np.array_equal(window, [[3, 4], [0, 0], [0, 0]]), 'Window crossed episode end or used wrong padding'
assert np.array_equal(mask, [False, True, True]), 'Mask must mark only the padded tail'

### Use your function

In [ ]:
print('window shape:', window.shape, 'valid steps:', int((~mask).sum()))

## Exercise 4: Split by complete episodes

**Purpose:** Keep validation episodes out of the training rows.

**Contract:** `split_episode_ids(ids, *, validation_fraction=0.2) -> (train_ids, val_ids)`

Input IDs are unique and nonnegative. Preserve input order and return nonempty disjoint partitions.

**Method:** Choose a validation suffix based on the requested fraction.

**Pointers:** A random row split leaks nearby frames into validation.

<details><summary>Detailed hint</summary>

Round the validation count, then clamp it to `[1, len(ids)-1]`.

</details>

Estimated time: about 15 minutes.

In [ ]:
def split_episode_ids(
    episode_ids: Sequence[int], *, validation_fraction: float = 0.2
) -> tuple[list[int], list[int]]:
    raise NotImplementedError('Exercise 3: split whole episodes')

### Check

In [ ]:
train_ids, val_ids = split_episode_ids([0, 1, 2, 3, 4], validation_fraction=0.2)
assert set(train_ids).isdisjoint(val_ids) and train_ids + val_ids == [0, 1, 2, 3, 4], 'Episodes must be split without overlap'
assert train_ids and val_ids, 'Both partitions must contain an episode'

### Use your function

In [ ]:
print('train episodes:', train_ids, 'validation episodes:', val_ids)

## Exercise 5: Fit training-only statistics

**Purpose:** Normalize model features without validation leakage.

**Contract:** `fit_training_statistics(train, *, validation_episodes=None) -> TrainingStats`

Mappings map episode IDs to arrays with the feature axis last. Return population mean/std and digest.

**Method:** Concatenate training episode samples only. Replace zero std with one for safe normalization.

**Pointers:** Pass validation to the contract so your check can prove it has no effect.

<details><summary>Detailed hint</summary>

Validation values can be extreme; the fitted result must stay unchanged.

</details>

Estimated time: about 15 minutes.

In [ ]:
def fit_training_statistics(
    training_episodes: Mapping[int, ArrayLike],
    *,
    validation_episodes: Mapping[int, ArrayLike] | None = None,
) -> TrainingStats:
    raise NotImplementedError('Exercise 4: fit statistics from training data only')

### Check

In [ ]:
stats = fit_training_statistics({0: [[0.0], [2.0]]}, validation_episodes={9: [[1000.0]]})
assert np.allclose(stats.mean, [1]) and np.allclose(stats.std, [1]), 'Validation rows affected training statistics'

### Use your function

In [ ]:
print('training mean/std:', stats.mean, stats.std)

## Larger experiment

Use the local dataset only when its pinned files already exist. Compare two whole-episode training subsets (2 versus 8 episodes) with the same validation episodes, fresh ACT initialization, chunk horizon, optimizer, seed, and 20-update budget. Your row ordering, action-window function, and train-only statistics function build the batches that enter the Unit 3 loss callback and optimizer loop. Compare validation error after converting both predictions and targets to physical action units.

Set `RUN_LOCAL_DATA=False` for the notebook default. The public reader loads every action/state row from at most eight frozen episodes, and decodes only four image frames per episode. The rows use individual 10 Hz actions; your window function creates the action chunks. No data download occurs. Training uses CUDA when available and CPU otherwise. Each subset records `completed` or `failed`; keep both rows. Both trials start from the same newly initialized pinned ACT weights, with no robot-trained checkpoint or pretrained vision backbone. This short run teaches data and batch effects, not policy success. It does not change the runner's one-action-per-controller-tick behavior.

In [ ]:
RUN_LOCAL_DATA = False
if RUN_LOCAL_DATA:
    import copy
    import yaml
    import torch
    from vla_lab.teaching.live import find_workspace, make_act_raw_model
    from vla_lab.teaching.data import read_local_episode_rows, build_act_batches
    from vla_lab.teaching.training import act_loss_fn, train_loop
    from lerobot.policies.act import ACTPolicy
    from vla_lab.policy.act import make_act_config
    root = find_workspace()
    task = yaml.safe_load((root / 'configs/data/episodes.yaml').read_text())['tasks']['bowl_plate']
    small_ids, large_ids = task['train_episode_ids'][:2], task['train_episode_ids'][:8]
    validation_ids = task['validation_episode_ids'][:2]
    results = {}
    loaded = {}

    def order_samples(episodes: dict[int, TrainingEpisode]) -> dict[int, TrainingEpisode]:
        keyed = [{'episode_index': eid, 'frame_index': sample['start'], '_episode': eid, '_sample': sample}
                 for eid, episode in episodes.items() for sample in episode['samples']]
        ordered = order_episode_rows(keyed)
        for eid, episode in episodes.items():
            episode['samples'] = [row['_sample'] for row in ordered if row['_episode'] == eid]
        return episodes

    try:
        small_raw = read_local_episode_rows('bowl_plate', small_ids, workspace=root,
                                            row_fn=extract_episode_arrays)
        large_raw = read_local_episode_rows('bowl_plate', large_ids, workspace=root,
                                            row_fn=extract_episode_arrays)
        validation_raw = read_local_episode_rows('bowl_plate', validation_ids, workspace=root,
                                                 row_fn=extract_episode_arrays)
        small_data = order_samples(small_raw)
        large_data = order_samples(large_raw)
        validation_data = order_samples(validation_raw)
        batches_small = build_act_batches(small_data, validation_data, horizon=20, batch_size=2,
                                          window_fn=action_window, stats_fn=fit_training_statistics)
        batches_large = build_act_batches(large_data, validation_data, horizon=20, batch_size=2,
                                          window_fn=action_window, stats_fn=fit_training_statistics)
        for built in (batches_small, batches_large):
            assert tuple(built['train'][0]['action'].shape) == (2, 20, 7), 'Learner windows did not reach ACT batches'
            assert tuple(built['train'][0]['observation.state'].shape) == (2, 8), 'Learner statistics did not reach ACT batches'
            assert built['train'][0]['action_is_pad'].dtype == torch.bool, 'Padding mask must reach the loss callback'
        # The pinned config disables pretrained backbone weights. Both trials
        # copy this exact fresh initialization; no prepared robot checkpoint is loaded.
        torch.manual_seed(17)
        base_policy = ACTPolicy(make_act_config(device='cpu', chunk_size=20)).cpu()
        initial_digest = hashlib.sha256()
        for name, value in sorted(base_policy.state_dict().items()):
            initial_digest.update(name.encode())
            initial_digest.update(value.detach().cpu().contiguous().numpy().tobytes())
        initial_weights_sha256 = initial_digest.hexdigest()
        loaded = {'two episodes': batches_small, 'eight episodes': batches_large}
    except Exception as error:
        message = f'{type(error).__name__}: {error}'
        results = {
            'two episodes': {'status': 'failed', 'error': message},
            'eight episodes': {'status': 'failed', 'error': message},
        }

    if loaded:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        def move_batches(built: ACTBatchData) -> ACTBatchData:
            return {**built,
                'train': [{key: value.to(device) for key, value in batch.items()} for batch in built['train']],
                'validation': [{key: value.to(device) for key, value in batch.items()} for batch in built['validation']]}
        loaded = {label: move_batches(built) for label, built in loaded.items()}
        for label, built in loaded.items():
            policy = model = optimizer = physical_validation_l1 = None
            try:
                torch.manual_seed(17)
                policy = copy.deepcopy(base_policy).to(device)
                model = make_act_raw_model(policy).to(device)
                optimizer = torch.optim.AdamW(policy.get_optim_params(), lr=1e-6)
                def physical_validation_l1(batch: Mapping[str, torch.Tensor]) -> torch.Tensor:
                    prediction = model(batch)[0]
                    mean = torch.as_tensor(built['action_stats'].mean, dtype=prediction.dtype, device=device)
                    std = torch.as_tensor(built['action_stats'].std, dtype=prediction.dtype, device=device)
                    prediction = prediction * std + mean
                    target = batch['action'] * std + mean
                    valid = (~batch['action_is_pad'].bool()).unsqueeze(-1).to(prediction.dtype)
                    return ((prediction - target).abs() * valid).sum() / (valid.sum() * prediction.shape[-1]).clamp_min(1)
                model.eval()
                with torch.no_grad():
                    initial_validation_l1 = [float(physical_validation_l1(batch).cpu()) for batch in built['validation']]
                history = train_loop(model, built['train'], optimizer, act_loss_fn, steps=20)
                model.eval()
                with torch.no_grad():
                    validation_l1 = [float(physical_validation_l1(batch).cpu()) for batch in built['validation']]
                results[label] = {'status': 'completed', 'device': str(device),
                    'train_episode_ids': built['training_episode_ids'],
                    'validation_episode_ids': built['validation_episode_ids'], 'updates': len(history),
                    'initial_weights_sha256': initial_weights_sha256,
                    'last_train_loss_normalized': history[-1]['loss'],
                    'initial_validation_action_l1_physical': sum(initial_validation_l1) / len(initial_validation_l1),
                    'validation_action_l1_physical': sum(validation_l1) / len(validation_l1)}
            except Exception as error:
                results[label] = {'status': 'failed', 'device': str(device),
                    'train_episode_ids': built['training_episode_ids'],
                    'validation_episode_ids': built['validation_episode_ids'],
                    'initial_weights_sha256': initial_weights_sha256,
                    'error': f'{type(error).__name__}: {error}'}
            finally:
                policy = model = optimizer = physical_validation_l1 = None
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
    assert set(results) == {'two episodes', 'eight episodes'}, 'Keep both subset attempts in the result table'
    for label, trial in results.items():
        if trial['status'] == 'completed':
            assert trial['updates'] == 20, f'{label} did not use the fixed 20-update budget'
    print(results)

## Record your result

Keep the inputs, every attempt, and the result table with the notebook. A failed run is a result and stays in the denominator.